In [0]:
%sql
SHOW VOLUMES IN bfsi_lakehouse.raw;

In [0]:

OUTPUT_PATH = "/Volumes/bfsi_lakehouse/raw/synthetic_data/"
for tbl in ["t_Client","t_AccountCustomer","t_Loan","t_LoanInstallment","t_AccountTrx"]:
    try:
        dbutils.fs.rm(f"{OUTPUT_PATH}/{tbl}/dt=2024-01-01", recurse=True)
        print(f"Cleaned {tbl}/dt=2024-01-01")
    except Exception as e:
        print(f"  (skipped {tbl}: {e})")

In [0]:
# Inspect partition structure
OUTPUT_PATH = "/Volumes/bfsi_lakehouse/raw/synthetic_data/"

display(dbutils.fs.ls(f"{OUTPUT_PATH}"))


In [0]:
# Row counts per table
for tbl in ["t_Client", "t_AccountCustomer", "t_Loan", "t_LoanInstallment", "t_AccountTrx"]:
    cnt = spark.read.parquet(f"{OUTPUT_PATH}/{tbl}").count()
    print(f"{tbl:<22} : {cnt:>14,}")

In [0]:
# Tabels schema
OUTPUT_PATH = "/Volumes/bfsi_lakehouse/raw/synthetic_data/"
for tbl in ["t_Client", "t_AccountCustomer", "t_Loan", "t_LoanInstallment", "t_AccountTrx"]:
    df = spark.read.parquet(f"{OUTPUT_PATH}/{tbl}")
    df.printSchema()

In [0]:
# Referential integrity checks (should all return 0)
acc  = spark.read.parquet(f"{OUTPUT_PATH}/t_AccountCustomer")
loan = spark.read.parquet(f"{OUTPUT_PATH}/t_Loan")
inst = spark.read.parquet(f"{OUTPUT_PATH}/t_LoanInstallment")

orphan_loans = loan.join(acc,  ["OurBranchID","AccountID"], "left_anti").count()
orphan_inst  = inst.join(loan, ["OurBranchID","AccountID","LoanSeries","LoanID"], "left_anti").count()
print(f"Orphan loans (no matching account):     {orphan_loans:,}  (expect 0 + a few from malformed AccountID DQ)")
print(f"Orphan installments (no matching loan): {orphan_inst:,}  (expect 0)")

In [0]:
from pyspark.sql import functions as F# DQ verification — confirm injection rates

print("DQ checks:")
print(f"  NULL State in t_Client      : {spark.read.parquet(f'{OUTPUT_PATH}/t_Client').filter(F.col('State').isNull()).count():,}")
print(f"  Duplicate LoanID in t_Loan  : {loan.groupBy('LoanID').count().filter(F.col('count') > 1).count():,}")
trx = spark.read.parquet(f"{OUTPUT_PATH}/t_AccountTrx")
print(f"  Malformed AccountID in trx  : {trx.filter(F.length('AccountID') != 10).count():,}")
print(f"  Future-dated CreatedAt      : {trx.filter(F.col('CreatedAt') > F.col('TrxDateTime')).count():,}")

In [0]:
# Volumn Size Check
def get_size(path):
/Volumes/bfsi_lakehouse/raw/synthetic_data/t_Client/dt=2024-01-05/    total = 0
    for item in dbutils.fs.ls(path):
        if item.isDir():
            total += get_size(item.path)
        else:
            total += item.size
    return total

size_bytes = get_size("/Volumes/bfsi_lakehouse/raw/synthetic_data/")
print(f"Size: {size_bytes / (1024**3):.2f} GB")

In [0]:
# acc  = spark.read.parquet(f"{OUTPUT_PATH}/t_AccountCustomer")
loan = spark.read.parquet(f"{OUTPUT_PATH}/t_Loan")
# inst = spark.read.parquet(f"{OUTPUT_PATH}/t_LoanInstallment")

loan.createOrReplaceTempView("loan")
spark.sql("""SELECT * FROM loan LIMIT 100""").display()